Show machine information for reproducibility:

In [55]:
versioninfo()

Julia Version 1.12.5
Commit 5fe89b8ddc1 (2026-02-09 16:05 UTC)
Build Info:
  Official https://julialang.org release
Platform Info:
  OS: macOS (arm64-apple-darwin24.0.0)
  CPU: 10 × Apple M1 Pro
  WORD_SIZE: 64
  LLVM: libLLVM-18.1.7 (ORCJIT, apple-m1)
  GC: Built with stock GC
Threads: 1 default, 1 interactive, 1 GC (on 8 virtual cores)


Activate environment:

In [56]:
using Pkg
Pkg.activate(pwd())
Pkg.instantiate()
# # add packages if needed
# Pkg.add(["BenchmarkTools"])
Pkg.status()

  Activating project at `~/Desktop/UCLA/Biostat 257/257-hw/hw1`


Status `~/Desktop/UCLA/Biostat 257/257-hw/hw1/Project.toml`
  [6e4b80f9] BenchmarkTools v1.7.0
  [7073ff75] IJulia v1.34.4
  [bdcacae8] LoopVectorization v0.12.173
  [91a5bcdd] Plots v1.41.6
  [f27b6e38] Polynomials v4.1.1
  [37e2e46d] LinearAlgebra v1.12.0
  [9a3f8284] Random v1.11.0


In [57]:
# Load packages
using BenchmarkTools, LinearAlgebra, Random, Statistics

## Q1. Git/GitHub

**No handwritten homework reports are accepted for this course.**  We work with Git/GitHub.  Efficient and abundant use of Git, e.g., **frequent and well-documented** commits, is an important criterion for grading your homework.

1. If you don't have a GitHub account, apply for the [Student Developer Pack](https://education.github.com/pack) at GitHub using your UCLA email.

2. Create a **private** repository `biostat-m257-2026-spring` and add `Hua-Zhou` and `toz015` (TA) as your collaborators.

3. Top directories of the repository should be `hw1`, `hw2`, ... You may create other branches for developing your homework solutions; but the `main` (or `master`) branch will be your presentation area. Put your homework submission files (Jupyter notebook `.ipynb`, html converted from notebook, `Project.toml`, all code and data set to reproduce results) in the `main` branch. 

4. After each homework due date, teaching assistant and instructor will check out your `main` branch for grading. Tag each of your homework submissions with tag names `hw1`, `hw2`, ...  Tagging time will be your submission time. That means if you tag your hw1 submission after deadline, penalty points will be deducted for late submission.  

5. Read the [style guide](https://github.com/invenia/BlueStyle) for Julia programming. Following rules in the style guide will be strictly enforced when grading: (1) four space indenting rule, (2) 92 charcter rule, (3) space after comma rule, (4) no space before comma rule, (5) space around binary operator rule.

## Q2. Computer arithmetics (18 points)

Let's check whether floating-point numbers obey certain algebraic rules. If a rule does not hold, one counter-example suffices. If a rule holds, briefly explain why it holds.

1. Associative rule for addition says `(x + y) + z == x + (y + z)`. Check association rule using `x = 0.1`, `y = 0.1` and `z = 1.0` in Julia. Explain what you find.

2. Do floating-point numbers obey the associative rule for multiplication: `(x * y) * z == x * (y * z)`?

3. Do floating-point numbers obey the distributive rule: `a * (x + y) == a * x + a * y`?  

4. Is `0 * x == 0` true for all floating-point number `x`? 

5. Is `x / a == x * (1 / a)` always true?

6. The commutative law for addition says `x + y = y + x` and the commutative law for multiplication says `x * y = y * x`. Do these two communtative laws hold for floating-point numbers? 

**Answer**


### Q2.1
We can see that the rule does not hold here. Because floating-point numbers are rounded after each operation. The two expressions group the additions differently, so rounding happens at different stages.

In [58]:
x, y, z = 0.1, 0.1, 1.0

println("(x + y) + z = ", (x + y) + z)
println("x + (y + z) = ", x + (y + z))
println("Equal? ", (x + y) + z == x + (y + z))  

(x + y) + z = 1.2
x + (y + z) = 1.2000000000000002
Equal? false


In [59]:
(x + (y + z)) - ((x + y) + z)

2.220446049250313e-16

In [60]:
(x + (y + z)) - ((x + y) + z) == eps(Float64)

true

### Q2.2
Floating-point multiplication is not always associative.

In [61]:
x, y, z = 0.1, 0.1, 0.3

println("(x * y) * z = ", (x * y) * z)
println("x * (y * z) = ", x * (y * z))
println("Equal? ", (x * y) * z == x * (y * z))

(x * y) * z = 0.0030000000000000005
x * (y * z) = 0.003
Equal? false


### Q2.3
Floating-point arithmetic does not always obey distributivity exactly.

In [62]:
a, x, y = 0.1, 0.1, 0.3

println(a * (x + y))
println(a * x + a * y)
println("Equal? ", a * (x + y) == a * x + a * y)

0.04000000000000001
0.04
Equal? false


### Q2.4
No.

In [63]:
println("0 * Infinity = ", 0 * Inf)
println("0 * NaN = ", 0 * NaN)

0 * Infinity = NaN
0 * NaN = NaN


### Q2.5
No.

In [64]:
x, a = 0.1, 1e-16
println("x / a = ", x / a)
println("x * (1 / a) = ", x * (1 / a))
println("Equal? ", x / a == x * (1 / a))

x / a = 1.0000000000000001e15
x * (1 / a) = 1.0e15
Equal? false


### Q2.6
Yes for ordinary numbers, but NaN makes == return false

In [65]:
println("NaN + 1 == 1 + NaN? ", NaN + 1 == 1 + NaN)
println("NaN * 2 == 2 * NaN? ", NaN * 2 == 2 * NaN)

NaN + 1 == 1 + NaN? false
NaN * 2 == 2 * NaN? false


## Q3. Multiple dispatch and JIT (18 points)

1. Consider Julia function
```julia
function g(k)
    for i in 1:10
        k = 5k - 1
    end
    k
end
```
a. Use `@code_llvm` to find the LLVM bitcode of compiled `g` with `Int64` input.   
b. Use `@code_llvm` to find the LLVM bitcode of compiled `g` with `Float64` input.  
c. Compare the bitcode from questions 1 and 2. Explain what do you find.  
d. Read Julia documentation on `@fastmath` and repeat the questions 1-3 on the function  

```julia
function g_fastmath(k)  
    @fastmath for i in 1:10  
        k = 5k - 1
    end
    k
end
```
e. Explain what does the macro `@fastmath` do? And why are the bitcodes for `g` and `g_fastmath` with `Float64` input different? (Hint: Q2)

2. Benchmark the following two codes
```julia
x = 1:2:10^8
@benchmark sum($x)
```
and
```julia
y = collect(1:2:10^8)
@benchmark sum($y)
```
Explain the difference in performance.

**Answer**


### Q3.1 a

In [66]:
function g(k)
    for i in 1:10
        k = 5k - 1
    end
    k
end

g (generic function with 1 method)

In [67]:
@code_llvm g(1)

; Function Signature: g(Int64)
;  @ In[66]:1 within `g`
define i64 @julia_g_15532(i64 signext %"k::Int64") #0 {
top:
;  @ In[66]:3 within `g`
; ┌ @ int.jl:88 within `*`
   %0 = mul i64 %"k::Int64", 9765625
; └
; ┌ @ int.jl:86 within `-`
   %1 = add i64 %0, -2441406
; └
;  @ In[66]:4 within `g`
  ret i64 %1
}


### Q3.1 b

In [68]:
@code_llvm g(1.0)

; Function Signature: g(Float64)
;  @ In[66]:1 within `g`
define double @julia_g_15534(double %"k::Float64") #0 {
top:
;  @ In[66]:3 within `g`
; ┌ @ promotion.jl:434 within `*` @ float.jl:497
   %0 = fmul double %"k::Float64", 5.000000e+00
; └
; ┌ @ promotion.jl:435 within `-` @ float.jl:496
   %1 = fadd double %0, -1.000000e+00
; └
; ┌ @ promotion.jl:434 within `*` @ float.jl:497
   %2 = fmul double %1, 5.000000e+00
; └
; ┌ @ promotion.jl:435 within `-` @ float.jl:496
   %3 = fadd double %2, -1.000000e+00
; └
; ┌ @ promotion.jl:434 within `*` @ float.jl:497
   %4 = fmul double %3, 5.000000e+00
; └
; ┌ @ promotion.jl:435 within `-` @ float.jl:496
   %5 = fadd double %4, -1.000000e+00
; └
; ┌ @ promotion.jl:434 within `*` @ float.jl:497
   %6 = fmul double %5, 5.000000e+00
; └
; ┌ @ promotion.jl:435 within `-` @ float.jl:496
   %7 = fadd double %6, -1.000000e+00
; └
; ┌ @ promotion.jl:434 within `*` @ float.jl:497
   %8 = fmul double %7, 5.000000e+00
; └
; ┌ @ promotion.jl:435 within `

### Q3.1 c
The two bitcodes are different. Julia uses multiple dispatch and JIT compilation, so it generates a separate compiled version of g for each concrete input type.

What we find is:
- g(::Int64) uses integer arithmetic instructions.
- g(::Float64) uses floating-point arithmetic instructions.
- Even though the Julia source code is the same, the compiled code differs because the argument types differ.

### Q3.1 d
- For Int64, the LLVM output is usually very similar to g(1), since @fastmath mainly affects floating-point operations.
- For Float64, the LLVM output is different from the ordinary g(1.0) version.

In [69]:
function g_fastmath(k)
    @fastmath for i in 1:10
        k = 5k - 1
    end
    k
end

g_fastmath (generic function with 1 method)

In [70]:
@code_llvm g_fastmath(1)

; Function Signature: g_fastmath(Int64)
;  @ In[69]:1 within `g_fastmath`
define i64 @julia_g_fastmath_15537(i64 signext %"k::Int64") #0 {
top:
;  @ In[69]:3 within `g_fastmath`
; ┌ @ fastmath.jl:277 within `mul_fast`
; │┌ @ int.jl:88 within `*`
    %0 = mul i64 %"k::Int64", 9765625
; └└
; ┌ @ fastmath.jl:263 within `sub_fast`
; │┌ @ int.jl:86 within `-`
    %1 = add i64 %0, -2441406
; └└
;  @ In[69]:4 within `g_fastmath`
  ret i64 %1
}


In [71]:
@code_llvm g_fastmath(1.0)

; Function Signature: g_fastmath(Float64)
;  @ In[69]:1 within `g_fastmath`
define double @julia_g_fastmath_15539(double %"k::Float64") #0 {
top:
;  @ In[69]:3 within `g_fastmath`
; ┌ @ fastmath.jl:274 within `mul_fast` @ fastmath.jl:169
   %0 = fmul fast double %"k::Float64", 0x4162A05F20000000
; └
; ┌ @ fastmath.jl:260 within `sub_fast` @ fastmath.jl:168
   %1 = fadd fast double %0, 0xC142A05F00000000
; └
;  @ In[69]:4 within `g_fastmath`
  ret double %1
}


### Q3.1 e
The macro @fastmath allows the compiler to perform more aggressive optimizations for floating-point arithmetic by relaxing strict IEEE floating-point rules. In particular, it may allow transformations that assume algebraic rules such as associativity or distributivity, even though these do not always hold exactly for floating-point numbers.

Therefore, g(1.0) must follow ordinary IEEE floating-point semantics, while g_fastmath(1.0) may use reordered or simplified floating-point operations. As a result, their LLVM bitcodes are different.

This connects to Q2: floating-point arithmetic is not always exactly associative or distributive, so allowing these optimizations can change the generated code and sometimes even the numerical result.

### Q3.2
x is a lazy range, while y is a fully allocated vector. Julia can sum a range using its arithmetic structure, but summing a vector requires looping over all elements in memory. Therefore, sum(x) is much faster and uses less memory than sum(y).

In [72]:
x = 1:2:10^8
@benchmark sum($x)

BenchmarkTools.Trial: 10000 samples with 1000 evaluations per sample.
 Range (min … max):  4.833 ns … 40.125 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     4.917 ns              ┊ GC (median):    0.00%
 Time  (mean ± σ):   4.977 ns ±  0.851 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

           ▃         █        ▅▃        ▁         ▂        ▂ ▁
  ▄▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁██▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁█ █
  4.83 ns      Histogram: log(frequency) by time     5.08 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [73]:
y = collect(1:2:10^8)
@benchmark sum($y)

BenchmarkTools.Trial: 696 samples with 1 evaluation per sample.
 Range (min … max):  6.826 ms …   9.918 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     7.083 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   7.170 ms ± 270.737 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

         ▂████▃▁                                               
  ▃▃▇▆▅▄▇███████▇▆▆▆▄▅▄▃▆▅▆▅▅▄▄▄▅▄▃▄▃▂▁▃▂▃▂▃▂▂▂▂▂▁▁▁▂▁▂▁▁▁▁▂▂ ▃
  6.83 ms         Histogram: frequency by time         8.1 ms <

 Memory estimate: 0 bytes, allocs estimate: 0.

## Q4. Sample variance (20 points)

Given a vector of numbers $x_1, \ldots, x_n$, we want to calculate the sample variance.

1. Implement the textbook algorithm for sample variance
$$
    s^2 = \frac{1}{n - 1} \sum_{i=1}^n (x_i - \bar{x})^2.
$$
How many floating-point operations are needed to compute the sample variance using this textbook (two-pass) algorithm?

2. Implement the one-pass algorithm for sample variance
$$
    s^2 = \frac{1}{n - 1} \left( \sum_{i=1}^n x_i^2 - n \bar{x}^2 \right).
$$
How many floating-point operations are needed to compute the sample variance using this one-pass algorithm?

3. Generate data by
```julia
Random.seed!(257)
x = rand(1_000_000) .+ 1e8;
```
and benchmark the two-pass and one-pass algorithms. Memory allocations should be zero. 

4. Display the results of the two-pass and one-pass algorithms on this test data. Any surprises? Explain what you found.

5. Investigate the algorithm implemented in the Julia function `var`. Is the `var` function the textbook (two-pass) algorithm or the one-pass algorithm? Display the result on the test data and benchmark it. Explain what you found.

**Answer**


### Q4.1
- first pass for the mean: n-1 additions and 1 division
- second pass: n subtractions, n multiplications, n-1 additions, and 1 division

So the total number of flops is $(n-1)+1+n+n+(n-1)+1 = 4n$.

Thus, the textbook two-pass algorithm uses $4n$ floating-point operations.

In [74]:
function var_twopass(x::AbstractVector{T}) where {T<:AbstractFloat}
    n = length(x)
    s = zero(T)
    @inbounds for i in eachindex(x)
        s += x[i]
    end
    xbar = s / n

    ss = zero(T)
    @inbounds for i in eachindex(x)
        d = x[i] - xbar
        ss += d * d
    end
    return ss / (n - 1)
end

var_twopass (generic function with 1 method)

### Q4.2
- in the loop: for each x_i, 1 multiplication and 2 additions, so 3n flops
- after the loop: 1 division, 2 multiplications, 1 subtraction, and 1 division, so 5 more flops

Hence the total is $3n+5 \approx 3n$

Thus, the one-pass algorithm uses $3n$ floating-point operations.

In [75]:
function var_onepass(x::AbstractVector{T}) where {T<:AbstractFloat}
    n = length(x)
    s1 = zero(T)
    s2 = zero(T)
    @inbounds for i in eachindex(x)
        xi = x[i]
        s1 += xi
        s2 += xi * xi
    end
    xbar = s1 / n
    return (s2 - n * xbar * xbar) / (n - 1)
end

var_onepass (generic function with 1 method)

### Q4.3
The one-pass algorithm is about twice as fast, which is consistent with its lower flop count.

In [76]:
Random.seed!(257)
x = rand(1_000_000) .+ 1e8

1000000-element Vector{Float64}:
 1.0000000052494614e8
 1.0000000080070095e8
 1.0000000047468221e8
 1.0000000079585788e8
 1.0000000074866852e8
 1.0000000091524786e8
 1.0000000099551688e8
 1.0000000050333282e8
 1.0000000009840925e8
 1.0000000054769354e8
 1.0000000018857381e8
 1.00000000462106e8
 1.0000000012038535e8
 ⋮
 1.0000000086151226e8
 1.000000008847299e8
 1.000000002827402e8
 1.0000000029220337e8
 1.0000000084380928e8
 1.0000000022721893e8
 1.0000000037091465e8
 1.0000000094119738e8
 1.0000000056018579e8
 1.0000000073921418e8
 1.0000000064604725e8
 1.0000000048441522e8

In [77]:
@benchmark var_twopass($x)

BenchmarkTools.Trial: 2021 samples with 1 evaluation per sample.
 Range (min … max):  2.451 ms …  2.892 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     2.461 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):   2.470 ms ± 26.516 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▁▄▆▆█▄▄▄▄▃▁▂▁▂▂▂▁▁▂▁▁▁                                     ▁
  █████████████████████████▇██▇██▇▁▆▅▆▆▄▄▁▄▁▁▄▅▁▄▄▅▁▄▁▄▁▁▁▁▄ █
  2.45 ms      Histogram: log(frequency) by time     2.57 ms <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [78]:
@benchmark var_onepass($x)

BenchmarkTools.Trial: 4019 samples with 1 evaluation per sample.
 Range (min … max):  1.226 ms …   8.111 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     1.231 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   1.242 ms ± 133.187 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █▂█▃▆▄▅▄▃▃▂▂▁▂▃▂▂▃▂▂▁▂▁▁▁   ▁                               ▂
  ████████████████████████████████▇▇▇▇▆▆▆▄▆▅▆▆▆▄▅▃▅▅▁▄▄▅▁▃▄▃▅ █
  1.23 ms      Histogram: log(frequency) by time       1.3 ms <

 Memory estimate: 0 bytes, allocs estimate: 0.

### Q4.4
This is very surprising, because a variance should never be negative. The two-pass result is reasonable. However, the one-pass algorithm gives -1033.9, which is completely wrong.

The reason is catastrophic cancellation. In the one-pass formula, $\sum x_i^2 - n\bar x^2$, both terms are extremely large because the data are around 10^8, and they are also very close to each other. Subtracting two large nearly equal floating-point numbers destroys many significant digits, so rounding error dominates the final answer. So although the one-pass algorithm is faster, it is much less reliable numerically on this data.

In [79]:
println("two-pass = ", var_twopass(x))
println("one-pass = ", var_onepass(x))

two-pass = 0.08335789995826702
one-pass = -1033.89696989697


### Q4.5
Julia’s official documentation states that for arrays, var(itr) is equivalent to
$\mathrm{sum}(\mathrm{abs2}.(itr .- \mathrm{mean}(itr))) / (n-1)$,
with the usual adjustment depending on corrected=true/false. This is a centered computation based on deviations from the mean, not the naive one-pass formula $\sum x_i^2 - n\bar{x}^2$.  ￼

So Julia’s var is effectively a two-pass centered algorithm for arrays. Its output on the test data, 0.083, matches the two-pass result and is far from the unstable one-pass value. This confirms both from documentation and from experiment that Julia’s var is using the numerically stable centered approach.  ￼

The benchmark also shows that Julia’s var is much faster than the hand-written two-pass function while still remaining stable. This is consistent with an optimized library implementation of the centered variance formula.

In [80]:
@benchmark var($x)

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  333.417 μs … 421.208 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     334.833 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   336.898 μs ±   5.894 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▄▇█▆▂▂▅▆▄                                                     ▂
  ████████████████▇██▇█████▇▇▇█▇▆▆▇▆▆▆▇▆▆▆▆▆▅▅▆▆▆▆▆▆▄▆▅▄▆▅▅▄▅▅▅ █
  333 μs        Histogram: log(frequency) by time        365 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [81]:
println("two-pass = ", var_twopass(x))
println("one-pass = ", var_onepass(x))
println("Julia var = ", var(x))

two-pass = 0.08335789995826702
one-pass = -1033.89696989697
Julia var = 0.08335789993927366


## Q5. Woodbury formula (12 points)

Demonstrate the following results in Julia (one numerical example for each fact). Mathematically curious ones are encouraged to prove them. 

1. **Sherman-Morrison formula**:
$$
	(\mathbf{A} + \mathbf{u} \mathbf{u}^T)^{-1} = \mathbf{A}^{-1} - \frac{1}{1 + \mathbf{u}^T \mathbf{A}^{-1} \mathbf{u}} \mathbf{A}^{-1} \mathbf{u} \mathbf{u}^T \mathbf{A}^{-1},
$$
where $\mathbf{A} \in \mathbb{R}^{n \times n}$ is nonsingular and $\mathbf{u} \in \mathbb{R}^n$. This formula supplies the inverse of the symmetric, rank-one  perturbation of $\mathbf{A}$.

2. **Woodbury formula**:
$$
	(\mathbf{A} + \mathbf{U} \mathbf{V}^T)^{-1} = \mathbf{A}^{-1} - \mathbf{A}^{-1} \mathbf{U} (\mathbf{I}_m + \mathbf{V}^T \mathbf{A}^{-1} \mathbf{U})^{-1} \mathbf{V}^T \mathbf{A}^{-1},
$$
where $\mathbf{A} \in \mathbb{R}^{n \times n}$ is nonsingular, $\mathbf{U}, \mathbf{V} \in \mathbb{R}^{n \times m}$, and $\mathbf{I}_m$ is the $m \times m$ identity matrix. In many applications $m$ is much smaller than $n$. Woodbury formula generalizes Sherman-Morrison and is valuable because the smaller matrix $\mathbf{I}_m + \mathbf{V}^T \mathbf{A}^{-1} \mathbf{U}$ is cheaper to invert than the larger matrix $\mathbf{A} + \mathbf{U} \mathbf{V}^T$.

3. **Binomial inversion formula**:
$$
	(\mathbf{A} + \mathbf{U} \mathbf{B} \mathbf{V}^T)^{-1} = \mathbf{A}^{-1} - \mathbf{A}^{-1} \mathbf{U} (\mathbf{B}^{-1} + \mathbf{V}^T \mathbf{A}^{-1} \mathbf{U})^{-1} \mathbf{V}^T \mathbf{A}^{-1},
$$
where $\mathbf{A}$ and $\mathbf{B}$ are nonsingular.

4. **Determinant identity**:
$$
	\text{det}(\mathbf{A} + \mathbf{U} \mathbf{V}^T) = \text{det}(\mathbf{A}) \text{det}(\mathbf{I}_m + \mathbf{V}^T \mathbf{A}^{-1} \mathbf{U}).
$$
This formula is useful for evaluating the density of a multivariate normal with covariance matrix $\mathbf{A} + \mathbf{U} \mathbf{V}^T$.

**Answer:** For all four identities, the left-hand side and right-hand side agree numerically up to floating-point error. This confirms the formulas in Julia for specific examples.


### Q5.1

In [82]:
n = 4
A = randn(n, n)
A = A'A + I           # make A symmetric positive definite
u = randn(n)

lhs1 = inv(A + u*u')
rhs1 = inv(A) - (inv(A)*u*u'*inv(A)) / (1 + u' * inv(A) * u)

rhs1 - lhs1


4×4 Matrix{Float64}:
  1.11022e-16  2.77556e-17  -2.77556e-17   5.55112e-17
  0.0          2.77556e-17   1.38778e-17   2.08167e-17
 -2.77556e-17  1.38778e-17   2.77556e-17   5.20417e-18
 -1.38778e-17  6.93889e-18  -3.46945e-18  -1.11022e-16

In [83]:
isapprox(lhs1, rhs1)

true

### Q5.2

In [84]:
n = 5
m = 2
A = randn(n, n)
A = A'A + I
U = randn(n, m)
V = randn(n, m)

Ainv = inv(A)
lhs2 = inv(A + U*V')
rhs2 = Ainv - Ainv*U*inv(I + V'*Ainv*U)*V'*Ainv

lhs2 - rhs2


5×5 Matrix{Float64}:
 -1.38778e-17   0.0          -4.16334e-17  -2.1684e-17   5.55112e-17
  9.97466e-18   5.55112e-17   8.32667e-17  -1.38778e-17  5.55112e-17
  2.77556e-17   0.0          -1.66533e-16  -8.32667e-17  2.22045e-16
 -1.73472e-18   2.42861e-17  -5.55112e-17  -1.38778e-17  2.08167e-17
  1.38778e-17  -6.245e-17     0.0          -2.77556e-17  0.0

In [85]:
isapprox(lhs2, rhs2)

true

### Q5.3

In [86]:
n = 5
m = 2
A = randn(n, n)
A = A'A + I
B = randn(m, m)
B = B'B + I
U = randn(n, m)
V = randn(n, m)

Ainv = inv(A)
Binv = inv(B)

lhs3 = inv(A + U*B*V')
rhs3 = Ainv - Ainv*U*inv(Binv + V'*Ainv*U)*V'*Ainv

lhs3 - rhs3

5×5 Matrix{Float64}:
  0.0           0.0           1.38778e-17   0.0          -5.55112e-17
  2.77556e-17   2.77556e-17   2.77556e-17   5.55112e-17   2.77556e-17
 -2.42861e-17  -1.38778e-17   0.0          -2.77556e-17  -1.21431e-17
 -5.55112e-17   1.38778e-17  -5.55112e-17  -5.55112e-17  -3.46945e-18
  4.16334e-17   1.04083e-17  -2.08167e-17  -3.46945e-17   0.0

In [87]:
isapprox(lhs3, rhs3)

true

### Q5.4

In [88]:
n = 5
m = 2
A = randn(n, n)
A = A'A + I
U = randn(n, m)
V = randn(n, m)

lhs4 = det(A + U*V')
rhs4 = det(A) * det(I + V' * inv(A) * U)

lhs4 - rhs4

-1.1368683772161603e-13

In [89]:
isapprox(lhs4, rhs4)

true

## Q6. Triangular matrix and orthogonal matrix (18 points)

Demonstrate the following facts about triangular and orthogonal matrices in Julia (one numerical example for each fact). Mathematically curious ones are encouraged to prove them. 

Note a unit triangular matrix is a triangular matrix with all diagonal entries being 1.

1. The product of two upper (lower) triangular matrices is upper (lower) triangular.

2. The inverse of an upper (lower) triangular matrix is upper (lower) triangular.

3. The product of two unit upper (lower) triangular matrices is unit upper (lower) triangular.

4. The inverse of a unit upper (lower) triangular matrix is unit upper (lower) triangular.

5. An orthogonal upper (lower) triangular matrix is diagonal. (You just need to prove this.)

6. The product of two orthogonal matrices is orthogonal.

**Answer:**


### Q6.1

In [90]:
# upper triangular
A = UpperTriangular(randn(4,4))
B = UpperTriangular(randn(4,4))
C = A * B

display(C)

4×4 UpperTriangular{Float64, Matrix{Float64}}:
 -2.22272  0.697229   2.89817   2.64312
   ⋅       1.7467     1.38112  -1.55386
   ⋅        ⋅        -1.63481   1.03976
   ⋅        ⋅          ⋅        0.0273035

In [91]:
# lower triangular
L1 = LowerTriangular(randn(4,4))
L2 = LowerTriangular(randn(4,4))
L3 = L1 * L2

display(L3)

4×4 LowerTriangular{Float64, Matrix{Float64}}:
  0.294807    ⋅          ⋅            ⋅ 
  1.74294   -0.0811453   ⋅            ⋅ 
 -0.749792   0.487546   0.00568377    ⋅ 
 -0.390611   3.64579    1.63904     -1.38113

### Q6.2

In [92]:
# upper triangular
A = UpperTriangular(randn(4,4) + 2I)
Ainv = inv(A)

display(Ainv)

4×4 UpperTriangular{Float64, Matrix{Float64}}:
 0.882178  -0.631205  -3.63737  -2.7197
  ⋅         0.447243   1.24842   1.25141
  ⋅          ⋅         4.44426   2.41249
  ⋅          ⋅          ⋅        0.533836

In [93]:
# lower triangular
A = LowerTriangular(randn(4,4) + 2I)
Ainv = inv(A)

display(Ainv)

4×4 LowerTriangular{Float64, Matrix{Float64}}:
  0.419405     ⋅          ⋅         ⋅ 
  0.152589    0.30571     ⋅         ⋅ 
 -0.147712   -0.0300841  0.298379   ⋅ 
  0.0247005  -0.056913   0.184183  0.671495

### Q6.3

In [94]:
# upper triangular
U1 = UpperTriangular(randn(4,4))
U2 = UpperTriangular(randn(4,4))
U1 = Matrix(U1); U2 = Matrix(U2)
U1[diagind(U1)] .= 1.0
U2[diagind(U2)] .= 1.0

Uprod = U1 * U2

display(Uprod)

4×4 Matrix{Float64}:
 1.0  0.221419  -0.0411984   0.0465152
 0.0  1.0        0.71673    -0.99698
 0.0  0.0        1.0        -1.40171
 0.0  0.0        0.0         1.0

In [95]:
# lower triangular
L1 = LowerTriangular(randn(4,4))
L2 = LowerTriangular(randn(4,4))
L1 = Matrix(L1); L2 = Matrix(L2)
L1[diagind(L1)] .= 1.0
L2[diagind(L2)] .= 1.0
Lprod = L1 * L2

display(Lprod)

4×4 Matrix{Float64}:
  1.0        0.0       0.0      0.0
 -1.26352    1.0       0.0      0.0
  0.117661  -1.34959   1.0      0.0
 -0.482289   0.759884  0.37938  1.0

### Q6.4

In [96]:
# upper triangular
U = UpperTriangular(randn(4,4))
U = Matrix(U)
U[diagind(U)] .= 1.0
Uinv = inv(U)

display(Uinv)

4×4 Matrix{Float64}:
 1.0  -1.52565   0.199704   2.07359
 0.0   1.0      -0.224566  -1.23205
 0.0   0.0       1.0        1.67581
 0.0   0.0       0.0        1.0

In [97]:
# lower triangular
L = LowerTriangular(randn(4,4))
L = Matrix(L)
L[diagind(L)] .= 1.0
Linv = inv(L)

display(Linv)

4×4 Matrix{Float64}:
  1.0        0.0        0.0      0.0
 -0.0113615  1.0        0.0      0.0
 -1.12652    0.266065   1.0      0.0
  2.19766    1.24895   -0.94602  1.0

### Q6.5

Let $Q$ be an upper triangular orthogonal matrix. Since $Q$ is orthogonal,
$$
Q^\top Q = I \rightarrow Q^\top = Q^{-1}
$$

Because $Q$ is upper triangular, $Q^\top$ is lower triangular, $Q^{-1}$ is upper triangular. Hence all off-diagonal entries for $Q^\top = Q^{-1}$ must be zero.

Therefore, $Q$ must be diagonal.

### Q6.6

Let $Q_1$ and $Q_2$ be orthogonal, so $Q_1^TQ_1 = I, \qquad Q_2^TQ_2 = I.$

Then
$$
(Q_1Q_2)^T(Q_1Q_2)
= Q_2^T Q_1^T Q_1 Q_2
= Q_2^T I Q_2
= Q_2^TQ_2
= I.
$$
So $Q_1Q_2$ is orthogonal.

In [98]:
A = randn(4,4)
B = randn(4,4)

Q1 = Matrix(qr(A).Q)
Q2 = Matrix(qr(B).Q)
Q = Q1 * Q2

println("Is Q1 orthogonal? ", Q1' * Q1 ≈ I)
println("Is Q2 orthogonal? ", Q2' * Q2 ≈ I)
println("Is Q1 * Q2 orthogonal? ", Q' * Q ≈ I)

Is Q1 orthogonal? true
Is Q2 orthogonal? true
Is Q1 * Q2 orthogonal? true


## Q7. AI assistant

Which AI assistants (e.g., GitHub Copilot) do you use when working on this assignment? Which AI model (e.g., GPT-5 mini, GPT-5, Claude Sonnet 4.5) does the AI assistant use? How do you use them? Do you think they help improve your productivity?

Give 5 instances where AI model gave incorrect or misleading answers. You can use screenshots or copy-paste the Q&A.

**Answer:** For this assignment, I mainly used ChatGPT as my AI assistant. The model I used was GPT-5.4 Thinking. I used it mainly to help with the following tasks:
- how to do the source control using VS code, 
- explaining Julia errors and debugging code,
- and helping me understand why certain algorithms behaved unexpectedly.
Overall, I think the AI assistant did improve my productivity. It saved time when I got stuck on syntax issues, package errors, and formatting.